# Mount Drive and connect GEE:

In [7]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
import ee

auth.authenticate_user()
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='geovisionpakistan')

print("Ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready


# Install libraries:

In [8]:
!pip install osmnx geopandas -q
import osmnx as ox
import geopandas as gpd
import os, time

print("Libraries ready")

Libraries ready


# Download OSM Graveyards


In [ ]:
import osmnx as ox
import geopandas as gpd
import os, time

os.makedirs(
    "/content/drive/MyDrive/GeoVision exports/graveyards",
    exist_ok=True
)

print("Downloading graveyards from OpenStreetMap...")
start = time.time()

# Download by city to avoid timeout
cities = [
    "Lahore, Pakistan",
    "Karachi, Pakistan",
    "Islamabad, Pakistan",
    "Rawalpindi, Pakistan",
    "Faisalabad, Pakistan",
    "Peshawar, Pakistan",
    "Quetta, Pakistan",
    "Multan, Pakistan",
]

all_gdfs = []

for city in cities:
    try:
        gdf = ox.features_from_place(
            city,
            tags={"landuse": "cemetery"}
        )
        gdf["city"] = city
        all_gdfs.append(gdf)
        print(f" {city}: {len(gdf)} graveyards")
    except Exception as e:
        print(f" {city}: failed")

# Combine all cities
import pandas as pd
combined = pd.concat(all_gdfs, ignore_index=True)
combined = gpd.GeoDataFrame(
    combined,
    geometry="geometry",
    crs="EPSG:4326"        # GPS degrees
)

# Convert to metres
combined = combined.to_crs("EPSG:32642")

# Save
out = "/content/drive/MyDrive/GeoVision exports/graveyards/locations.geojson"
combined.to_file(out, driver="GeoJSON")

elapsed = time.time() - start
mins, secs = divmod(int(elapsed), 60)
print(f"\nTotal: {len(combined)} graveyards")
print(f"Time : {mins:02d}m {secs:02d}s")
print(f"Saved: {out}")

 Lahore, Pakistan: failed
 Karachi, Pakistan: 144 graveyards
 Islamabad, Pakistan: 23 graveyards
 Rawalpindi, Pakistan: failed
 Faisalabad, Pakistan: 15 graveyards
 Peshawar, Pakistan: 29 graveyards
 Quetta, Pakistan: 17 graveyards
 Multan, Pakistan: failed

Total: 228 graveyards
Time : 01m 27s
Saved: /content/drive/MyDrive/GeoVision exports/graveyards/locations.geojson


# Retry Failed Cities With Bbox

In [ ]:
import requests
import pandas as pd
import time

def get_graveyards_fast(city_name, lat_min, lat_max, lon_min, lon_max):

    query = f"""
    [out:json][timeout:60];
    (
      way["landuse"="cemetery"]
      ({lat_min},{lon_min},{lat_max},{lon_max});
    );
    out center;
    """

    # Try multiple servers
    servers = [
        "https://overpass.kumi.systems/api/interpreter",
        "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
        "https://overpass.openstreetmap.ru/api/interpreter",
    ]

    for server in servers:
        try:
            print(f"  Trying {server[:40]}...")
            response = requests.post(
                server,
                data=query,
                timeout=60
            )
            if response.status_code == 200:
                data     = response.json()
                elements = data.get("elements", [])
                points   = []

                for el in elements:
                    if "center" in el:
                        points.append({
                            "city"  : city_name,
                            "lat"   : el["center"]["lat"],
                            "lon"   : el["center"]["lon"],
                            "osm_id": el["id"]
                        })

                print(f" {city_name}: {len(points)} graveyards")
                return pd.DataFrame(points) if points else None

        except Exception as e:
            print(f"  Failed: {e}")
            time.sleep(5)
            continue

    print(f" {city_name}: all servers failed")
    return None

# Retry failed cities
failed = {
    "Lahore"    : (31.40, 31.70, 74.20, 74.50),
    "Rawalpindi": (33.50, 33.70, 72.90, 73.20),
    "Multan"    : (30.10, 30.30, 71.30, 71.60),
}

all_dfs = []
for city, bbox in failed.items():
    df = get_graveyards_fast(city, *bbox)
    if df is not None:
        all_dfs.append(df)
    time.sleep(10)  # wait between requests

if all_dfs:
    retry_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\nRetry total: {len(retry_df)} graveyards")
else:
    print("All servers failed - will use 228 graveyards we have")

  Trying https://overpass.kumi.systems/api/interp...
  Trying https://maps.mail.ru/osm/tools/overpass/...
  Trying https://overpass.openstreetmap.ru/api/in...
  Failed: HTTPSConnectionPool(host='overpass.openstreetmap.ru', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x79a29ef1fd10>, 'Connection to overpass.openstreetmap.ru timed out. (connect timeout=60)'))
 Lahore: all servers failed
  Trying https://overpass.kumi.systems/api/interp...
  Trying https://maps.mail.ru/osm/tools/overpass/...
  Trying https://overpass.openstreetmap.ru/api/in...
  Failed: HTTPSConnectionPool(host='overpass.openstreetmap.ru', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x79a29ef1c740>, 'Connection to overpass.openstreetmap.ru timed out. (connect timeout=60)'))
 Rawalpindi: all servers failed
  Trying https://overpass.kumi.syste

In [ ]:
from shapely.geometry import Point
import geopandas as gpd
import json

# Load original 228
original = gpd.read_file(
    "/content/drive/MyDrive/GeoVision exports/graveyards/locations.geojson"
)

# Convert retry_df to GeoDataFrame
geometry = [Point(row.lon, row.lat) for _, row in retry_df.iterrows()]
retry_gdf = gpd.GeoDataFrame(
    retry_df,
    geometry=geometry,
    crs="EPSG:4326"
).to_crs("EPSG:32642")

# Keep only needed columns
retry_gdf = retry_gdf[["city", "geometry"]]

# Combine
import pandas as pd
final = pd.concat([original, retry_gdf], ignore_index=True)
final = gpd.GeoDataFrame(final, geometry="geometry", crs="EPSG:32642")
final = final.drop_duplicates(subset="geometry")

# Save
out = "/content/drive/MyDrive/GeoVision exports/graveyards/locations_final.geojson"
final.to_file(out, driver="GeoJSON")

print(f"Total graveyards saved: {len(final)}")
print(f"Saved to: {out}")

NameError: name 'retry_df' is not defined

# The Patch Extraction Code


In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
from PIL import Image
import geopandas as gpd
import os, time

# Load graveyard locations
gdf = gpd.read_file(
    "/content/drive/MyDrive/GeoVision exports/graveyards/locations_final.geojson"
)

# Output folders
pos_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/positive"
neg_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/negative"
os.makedirs(pos_dir, exist_ok=True)
os.makedirs(neg_dir, exist_ok=True)

# Settings
PATCH_SIZE_M = 6400   # 640 pixels × 10 metres
HALF         = PATCH_SIZE_M / 2

# Use your best S2 tile (highest resolution)
s2_path = "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000000000-0000000000.tif"

saved   = 0
skipped = 0
start   = time.time()

with rasterio.open(s2_path) as src:

    for idx, row in gdf.iterrows():

        # Get center point
        cx = row.geometry.centroid.x
        cy = row.geometry.centroid.y

        # Calculate corners
        left   = cx - HALF
        right  = cx + HALF
        bottom = cy - HALF
        top    = cy + HALF

        try:
            # Define window from bounds
            window = from_bounds(
                left, bottom, right, top,
                src.transform
            )

            # Read RGB bands only (for YOLOv8)
            patch = src.read([3, 2, 1], window=window)

            # Check patch size is correct
            if patch.shape[1] < 100 or patch.shape[2] < 100:
                skipped += 1
                continue

            # Check validity
            if np.sum(patch == 0) / patch.size > 0.30:
                skipped += 1
                continue

            # Normalize to 0-255 for saving as image
            patch = np.clip(patch / 3000.0 * 255, 0, 255)
            patch = patch.astype(np.uint8)

            # Save as JPG
            img = Image.fromarray(patch.transpose(1, 2, 0))
            img = img.resize((640, 640))
            img.save(f"{pos_dir}/graveyard_{saved:04d}.jpg")

            saved += 1

            if saved % 50 == 0:
                elapsed = time.time() - start
                mins, secs = divmod(int(elapsed), 60)
                print(f"[{mins:02d}:{secs:02d}] Saved {saved} patches")

        except Exception as e:
            skipped += 1
            continue

print(f"\nFinished")
print(f"Saved  : {saved} positive patches")
print(f"Skipped: {skipped} patches")


Finished
Saved  : 0 positive patches
Skipped: 339 patches


# The Fix — Search All 4 Tiles

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
from PIL import Image
import geopandas as gpd
import os, time

# Load graveyard locations
gdf = gpd.read_file(
    "/content/drive/MyDrive/GeoVision exports/graveyards/locations_final.geojson"
)

# Output folder
pos_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/positive"
os.makedirs(pos_dir, exist_ok=True)

# ALL 4 Sentinel-2 tiles
s2_tiles = [
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000000000-0000000000.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000000000-0000009472.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000009472-0000000000.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000009472-0000009472.tif",
]

PATCH_SIZE_M = 6400
HALF         = PATCH_SIZE_M / 2
saved        = 0
skipped      = 0
start        = time.time()

# For each graveyard location
for idx, row in gdf.iterrows():

    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y

    left   = cx - HALF
    right  = cx + HALF
    bottom = cy - HALF
    top    = cy + HALF

    patch_saved = False

    # Try each S2 tile until one works
    for s2_path in s2_tiles:

        try:
            with rasterio.open(s2_path) as src:

                # Check if this graveyard falls inside this tile
                tile_bounds = src.bounds
                if not (tile_bounds.left < cx < tile_bounds.right and
                        tile_bounds.bottom < cy < tile_bounds.top):
                    continue  # not in this tile, try next

                # Cut the patch
                window = from_bounds(
                    left, bottom, right, top,
                    src.transform
                )

                patch = src.read([3, 2, 1], window=window)

                # Validity checks
                if patch.shape[1] < 100 or patch.shape[2] < 100:
                    continue

                if np.sum(patch == 0) / patch.size > 0.30:
                    continue

                # Normalize and save
                patch = np.clip(patch / 3000.0 * 255, 0, 255)
                patch = patch.astype(np.uint8)

                img = Image.fromarray(patch.transpose(1, 2, 0))
                img = img.resize((640, 640))
                img.save(f"{pos_dir}/graveyard_{saved:04d}.jpg")

                saved       += 1
                patch_saved  = True
                break  # found the right tile, stop searching

        except Exception as e:
            continue

    if not patch_saved:
        skipped += 1

    # Progress every 50
    if (idx + 1) % 50 == 0:
        elapsed = time.time() - start
        mins, secs = divmod(int(elapsed), 60)
        print(f"[{mins:02d}:{secs:02d}] Processed {idx+1}/339 | Saved: {saved} | Skipped: {skipped}")

print(f"\nFinished")
print(f"Saved  : {saved} positive patches")
print(f"Skipped: {skipped} patches")

[00:44] Processed 50/339 | Saved: 0 | Skipped: 50
[01:21] Processed 100/339 | Saved: 0 | Skipped: 100
[01:59] Processed 150/339 | Saved: 0 | Skipped: 150
[02:41] Processed 200/339 | Saved: 0 | Skipped: 200
[03:24] Processed 250/339 | Saved: 0 | Skipped: 250
[04:13] Processed 300/339 | Saved: 0 | Skipped: 300

Finished
Saved  : 0 positive patches
Skipped: 339 patches


# Diagnostic Cell

In [ ]:
import rasterio
import geopandas as gpd

# Load graveyards
gdf = gpd.read_file(
    "/content/drive/MyDrive/GeoVision exports/graveyards/locations_final.geojson"
)

print("=== GRAVEYARD COORDINATES ===")
print(f"CRS: {gdf.crs}")
print(f"First 3 locations:")
for idx, row in gdf.head(3).iterrows():
    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y
    print(f"  x={cx:.2f}, y={cy:.2f}")

print()

# Check each S2 tile boundary
s2_tiles = [
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000000000-0000000000.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000000000-0000009472.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000009472-0000000000.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000009472-0000009472.tif",
]

print("=== SENTINEL-2 TILE BOUNDARIES ===")
for path in s2_tiles:
    name = path.split("/")[-1]
    with rasterio.open(path) as src:
        b = src.bounds
        print(f"{name}:")
        print(f"  CRS   : {src.crs}")
        print(f"  Left  : {b.left:.2f}")
        print(f"  Right : {b.right:.2f}")
        print(f"  Bottom: {b.bottom:.2f}")
        print(f"  Top   : {b.top:.2f}")
        print()

=== GRAVEYARD COORDINATES ===
CRS: EPSG:32642
First 3 locations:
  x=305673.77, y=2757199.95
  x=305614.02, y=2756847.57
  x=305746.07, y=2756829.11

=== SENTINEL-2 TILE BOUNDARIES ===
sentinel2_pakistan-0000000000-0000000000.tif:
  CRS   : EPSG:32642
  Left  : -331200.00
  Right : 616000.00
  Bottom: 3194500.00
  Top   : 4141700.00

sentinel2_pakistan-0000000000-0000009472.tif:
  CRS   : EPSG:32642
  Left  : 616000.00
  Right : 1404200.00
  Bottom: 3194500.00
  Top   : 4141700.00

sentinel2_pakistan-0000009472-0000000000.tif:
  CRS   : EPSG:32642
  Left  : -331200.00
  Right : 616000.00
  Bottom: 2637000.00
  Top   : 3194500.00

sentinel2_pakistan-0000009472-0000009472.tif:
  CRS   : EPSG:32642
  Left  : 616000.00
  Right : 1404200.00
  Bottom: 2637000.00
  Top   : 3194500.00



# Debug Cell

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
import geopandas as gpd

gdf = gpd.read_file(
    "/content/drive/MyDrive/GeoVision exports/graveyards/locations_final.geojson"
)

# Test just first graveyard on Tile 3
s2_path = "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000009472-0000000000.tif"

row = gdf.iloc[0]
cx  = row.geometry.centroid.x
cy  = row.geometry.centroid.y

print(f"Graveyard center: x={cx:.2f}, y={cy:.2f}")

HALF = 3200

left   = cx - HALF
right  = cx + HALF
bottom = cy - HALF
top    = cy + HALF

print(f"Patch bounds:")
print(f"  left={left:.2f}, right={right:.2f}")
print(f"  bottom={bottom:.2f}, top={top:.2f}")

with rasterio.open(s2_path) as src:

    tb = src.bounds
    print(f"\nTile bounds:")
    print(f"  left={tb.left:.2f}, right={tb.right:.2f}")
    print(f"  bottom={tb.bottom:.2f}, top={tb.top:.2f}")

    # Check if inside
    inside = (tb.left < cx < tb.right and
              tb.bottom < cy < tb.top)
    print(f"\nGraveyard inside tile: {inside}")

    if inside:
        window = from_bounds(
            left, bottom, right, top,
            src.transform
        )
        print(f"\nWindow: {window}")

        patch = src.read([3, 2, 1], window=window)
        print(f"Patch shape: {patch.shape}")
        print(f"Patch min: {patch.min()}")
        print(f"Patch max: {patch.max()}")
        print(f"Zero ratio: {np.sum(patch==0)/patch.size:.3f}")

Graveyard center: x=305673.77, y=2757199.95
Patch bounds:
  left=302473.77, right=308873.77
  bottom=2753999.95, top=2760399.95

Tile bounds:
  left=-331200.00, right=616000.00
  bottom=2637000.00, top=3194500.00

Graveyard inside tile: True

Window: Window(col_off=np.float64(6336.737693978927), row_off=np.float64(4341.000542704956), width=np.float64(64.0), height=np.float64(64.0))
Patch shape: (3, 64, 64)
Patch min: 440.3333333333333
Patch max: 3670.6666666666665
Zero ratio: 0.000


# Fixed Patch Extraction Code

In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
from PIL import Image
import geopandas as gpd
import os, time

gdf = gpd.read_file(
    "/content/drive/MyDrive/GeoVision exports/graveyards/locations_final.geojson"
)

pos_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/positive"
os.makedirs(pos_dir, exist_ok=True)

s2_tiles = [
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000000000-0000000000.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000000000-0000009472.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000009472-0000000000.tif",
    "/content/drive/MyDrive/GeoVision exports/sentinel2_pakistan-0000009472-0000009472.tif",
]

# FIXED: 6.4km patch = 64 pixels at 100m resolution
PATCH_SIZE_M = 6400
HALF         = PATCH_SIZE_M / 2

saved   = 0
skipped = 0
start   = time.time()

for idx, row in gdf.iterrows():

    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y

    left   = cx - HALF
    right  = cx + HALF
    bottom = cy - HALF
    top    = cy + HALF

    patch_saved = False

    for s2_path in s2_tiles:
        try:
            with rasterio.open(s2_path) as src:

                # Check if graveyard inside this tile
                tb = src.bounds
                if not (tb.left < cx < tb.right and
                        tb.bottom < cy < tb.top):
                    continue

                # Cut patch
                window = from_bounds(
                    left, bottom, right, top,
                    src.transform
                )

                patch = src.read([3, 2, 1], window=window)

                # Must have some valid data
                if patch.size == 0:
                    continue

                if np.sum(patch == 0) / patch.size > 0.30:
                    continue

                # Normalize to 0-255
                patch = np.clip(patch / 3000.0 * 255, 0, 255)
                patch = patch.astype(np.uint8)

                # Resize to 640×640 for YOLOv8
                img = Image.fromarray(patch.transpose(1, 2, 0))
                img = img.resize((640, 640),
                                 Image.NEAREST)
                img.save(f"{pos_dir}/graveyard_{saved:04d}.jpg")

                saved      += 1
                patch_saved = True
                break

        except Exception as e:
            continue

    if not patch_saved:
        skipped += 1

    if (idx + 1) % 50 == 0:
        elapsed = time.time() - start
        mins, secs = divmod(int(elapsed), 60)
        print(f"[{mins:02d}:{secs:02d}] {idx+1}/339 | Saved: {saved} | Skipped: {skipped}")

print(f"\nFinished")
print(f"Saved  : {saved} positive patches")
print(f"Skipped: {skipped} patches")

[00:38] 50/339 | Saved: 50 | Skipped: 0
[01:11] 100/339 | Saved: 100 | Skipped: 0
[01:42] 150/339 | Saved: 150 | Skipped: 0
[02:19] 200/339 | Saved: 200 | Skipped: 0
[02:58] 250/339 | Saved: 250 | Skipped: 0
[03:19] 300/339 | Saved: 300 | Skipped: 0

Finished
Saved  : 339 positive patches
Skipped: 0 patches


#  Create Annotation Files

In [ ]:
import os

pos_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/positive"

# Get all image files
images = sorted([f for f in os.listdir(pos_dir)
                 if f.endswith('.jpg')])

for img_file in images:

    # Create matching .txt filename
    txt_file = img_file.replace('.jpg', '.txt')
    txt_path = os.path.join(pos_dir, txt_file)

    # Write YOLO annotation
    # class cx cy width height
    with open(txt_path, 'w') as f:
        f.write("0 0.50 0.50 0.40 0.40\n")

print(f"Created {len(images)} annotation files")

# Verify first annotation
first_txt = os.path.join(pos_dir, images[0].replace('.jpg','.txt'))
with open(first_txt) as f:
    print(f"Sample annotation: {f.read()}")

Created 339 annotation files
Sample annotation: 0 0.50 0.50 0.40 0.40



# Augmentation Code

In [ ]:
import os
import numpy as np
from PIL import Image
import time

pos_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/positive"

images  = sorted([f for f in os.listdir(pos_dir)
                  if f.endswith('.jpg')])

start   = time.time()
created = 0

for img_file in images:

    img_path = os.path.join(pos_dir, img_file)
    base     = img_file.replace('.jpg', '')
    img      = Image.open(img_path)
    img_arr  = np.array(img)

    # Augmentation 1 — Horizontal flip
    flipped_h     = np.fliplr(img_arr)
    Image.fromarray(flipped_h).save(
        f"{pos_dir}/{base}_fliph.jpg")
    # Annotation: cx flips → 1 - 0.50 = 0.50
    # cy, width, height stay same
    with open(f"{pos_dir}/{base}_fliph.txt", 'w') as f:
        f.write("0 0.50 0.50 0.40 0.40\n")

    # Augmentation 2 — Vertical flip
    flipped_v     = np.flipud(img_arr)
    Image.fromarray(flipped_v).save(
        f"{pos_dir}/{base}_flipv.jpg")
    # Annotation stays same for vertical flip
    with open(f"{pos_dir}/{base}_flipv.txt", 'w') as f:
        f.write("0 0.50 0.50 0.40 0.40\n")

    # Augmentation 3 — Rotate 90 degrees
    rotated       = np.rot90(img_arr)
    Image.fromarray(rotated).save(
        f"{pos_dir}/{base}_rot90.jpg")
    # Annotation stays same (center rotation)
    with open(f"{pos_dir}/{base}_rot90.txt", 'w') as f:
        f.write("0 0.50 0.50 0.40 0.40\n")

    created += 3

elapsed    = time.time() - start
mins, secs = divmod(int(elapsed), 60)

print(f"Original patches : 339")
print(f"New patches added: {created}")
print(f"Total patches    : {339 + created}")
print(f"Time             : {mins:02d}m {secs:02d}s")

Original patches : 339
New patches added: 1017
Total patches    : 1356
Time             : 00m 40s


# Confirm your data is safe:

In [9]:
import geopandas as gpd
import os

base = "/content/drive/MyDrive/GeoVision exports"

# Check all important files
checks = {
    "Task A images"    : f"{base}/tiles/images",
    "Task A masks"     : f"{base}/tiles/masks",
    "Positive patches" : f"{base}/graveyards/patches/positive",
    "Graveyard locations": f"{base}/graveyards/locations_final.geojson",
}

for name, path in checks.items():
    if os.path.exists(path):
        if os.path.isdir(path):
            count = len(os.listdir(path))
            print(f"✓ {name}: {count} files")
        else:
            print(f"✓ {name}: exists")
    else:
        print(f"✗ {name}: NOT FOUND")

✓ Task A images: 11306 files
✓ Task A masks: 11305 files
✓ Positive patches: 2712 files
✓ Graveyard locations: exists



Task A images: 11,306 files
Task A masks:  11,305 files

These should be equal.
One extra image has no matching mask.
Difference = 1 file

Positive patches: 2,712 files
Should be 1,356 images + 1,356 txt = 2,712
This is correct.

TO FIX LATER:
Task A images = 11,306
Task A masks  = 11,305
Find and delete the extra image
that has no matching mask

Also — Earlier You Had 19,912 Tiles
Before:  19,912 tiles
Now:     11,306 tiles

8,606 tiles are missing.

Most likely reason:

Colab restarted during tiling
Tiles saved up to 11,306
then session crashed
Rest were lost

This means your Task A dataset is incomplete. But 11,306 tiles is still enough to train a good model. We continue with this.


# Check Free Disk Space

In [10]:
import shutil

total, used, free = shutil.disk_usage("/content")
print(f"Total: {total/1e9:.1f} GB")
print(f"Used : {used/1e9:.1f} GB")
print(f"Free : {free/1e9:.1f} GB")

Total: 120.9 GB
Used : 46.1 GB
Free : 74.9 GB


# Negative Patchs


Why 200 Instead of 2000

Reading from Drive directly:
200 files × ~5 seconds each = ~17 minutes
Manageable without GPU timeout

2000 files × ~5 seconds each = ~3 hours
Colab will disconnect again

Strategy:
Run 200 now
Run another 200 later
Build up to 2000 gradually
Each run saves to Drive permanently;

Actually — Even Smarter Strategy

You already have 1,356 positive patches.
Research shows ratio of:
1 positive : 2 negatives is enough

1,356 × 2 = 2,712 negatives needed

But even 1:1 ratio works for starting:
1,356 positives : 1,356 negatives

Start with 500 negatives first.
Train YOLOv8.
See results.
Add more negatives if needed.

In [12]:
import numpy as np
import os, random, time
from PIL import Image

src_dir = "/content/drive/MyDrive/GeoVision exports/tiles/images"
neg_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/negative"
os.makedirs(neg_dir, exist_ok=True)

all_tiles = sorted([
    f for f in os.listdir(src_dir)
    if f.endswith('.npy')
])

print(f"Available tiles: {len(all_tiles)}")

# Only sample 200 negatives now
# NOT 2000 — small batch to avoid timeout
random.seed(99)
selected = random.sample(all_tiles, 200)

saved = 0
start = time.time()

for filename in selected:

    img = np.load(f"{src_dir}/{filename}")
    rgb = img[[2, 1, 0], :, :]
    rgb = np.clip(rgb / 3000.0 * 255, 0, 255)
    rgb = rgb.astype(np.uint8)

    pil_img = Image.fromarray(rgb.transpose(1, 2, 0))
    pil_img = pil_img.resize((640, 640), Image.NEAREST)
    pil_img.save(f"{neg_dir}/negative_{saved:04d}.jpg")

    with open(f"{neg_dir}/negative_{saved:04d}.txt", 'w') as f:
        f.write("")

    saved += 1

    if saved % 50 == 0:
        elapsed    = time.time() - start
        mins, secs = divmod(int(elapsed), 60)
        print(f"[{mins:02d}:{secs:02d}] {saved}/200 saved")

print(f"Finished: {saved} negatives saved")

Available tiles: 11306
[00:34] 50/200 saved
[01:10] 100/200 saved
[01:39] 150/200 saved
[02:00] 200/200 saved
Finished: 200 negatives saved


Before: copy 11,306 files = 30 minutes
Now:    read only 200 files = 2 minutes

Lesson learned:
Never copy everything
Only read exactly what you need

Before:
→ You manually change offset 9 times
→ Run cell 9 times
→ Easy to make mistakes

Now:
→ One cell runs automatically
→ Loop handles all 9 batches
→ Saves negatives 0200 to 1999
→ Takes ~18 minutes total
→ No manual work needed

In [13]:
import numpy as np
import os, random, time
from PIL import Image

src_dir = "/content/drive/MyDrive/GeoVision exports/tiles/images"
neg_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/negative"
os.makedirs(neg_dir, exist_ok=True)

all_tiles = sorted([
    f for f in os.listdir(src_dir)
    if f.endswith('.npy')
])

print(f"Available tiles: {len(all_tiles)}")

# Total we want = 2000
# Already have = 200
# Still need   = 1800

BATCH_SIZE   = 200
START_OFFSET = 200    # already saved 0-199
TOTAL_NEEDED = 1800

random.seed(99)

# Generate all 1800 in batches of 200
total_saved = 0
batch_num   = 0

for offset in range(START_OFFSET,
                    START_OFFSET + TOTAL_NEEDED,
                    BATCH_SIZE):

    batch_num += 1
    print(f"\nBatch {batch_num}/9 | offset={offset}")

    # Fresh random sample for each batch
    selected = random.sample(all_tiles, BATCH_SIZE)

    batch_saved = 0
    start       = time.time()

    for filename in selected:

        img = np.load(f"{src_dir}/{filename}")
        rgb = img[[2, 1, 0], :, :]
        rgb = np.clip(rgb / 3000.0 * 255, 0, 255)
        rgb = rgb.astype(np.uint8)

        pil_img = Image.fromarray(rgb.transpose(1, 2, 0))
        pil_img = pil_img.resize((640, 640), Image.NEAREST)

        # Filename uses global offset
        save_idx = offset + batch_saved
        pil_img.save(f"{neg_dir}/negative_{save_idx:04d}.jpg")

        with open(f"{neg_dir}/negative_{save_idx:04d}.txt", 'w') as f:
            f.write("")

        batch_saved  += 1
        total_saved  += 1

    elapsed    = time.time() - start
    mins, secs = divmod(int(elapsed), 60)
    print(f"Batch {batch_num} done in {mins:02d}m {secs:02d}s")

print(f"\n{'='*40}")
print(f"All batches complete")
print(f"Total negatives saved: {START_OFFSET + total_saved}")

# Final count check
total_files = len([
    f for f in os.listdir(neg_dir)
    if f.endswith('.jpg')
])
print(f"JPG files in folder  : {total_files}")

Available tiles: 11306

Batch 1/9 | offset=200
Batch 1 done in 00m 04s

Batch 2/9 | offset=400


/tmp/ipykernel_2084/4281707221.py:48: RuntimeWarning: invalid value encountered in cast
  rgb = rgb.astype(np.uint8)


Batch 2 done in 01m 26s

Batch 3/9 | offset=600
Batch 3 done in 01m 23s

Batch 4/9 | offset=800
Batch 4 done in 01m 08s

Batch 5/9 | offset=1000
Batch 5 done in 01m 03s

Batch 6/9 | offset=1200
Batch 6 done in 01m 09s

Batch 7/9 | offset=1400
Batch 7 done in 01m 08s

Batch 8/9 | offset=1600
Batch 8 done in 01m 07s

Batch 9/9 | offset=1800
Batch 9 done in 01m 05s

All batches complete
Total negatives saved: 2000
JPG files in folder  : 2000


About That Warning

RuntimeWarning: invalid value encountered in cast
rgb = rgb.astype(np.uint8)
This means some pixels have value nan (Not a Number). When you try to convert nan to uint8 it produces unpredictable values.
Not a crash — just a warning. But worth noting.

Why nan Values Appear

Your Sentinel-2 data has some pixels
where the satellite had no valid reading.
These get stored as nan instead of 0.

Your validity check catches zeros
but NOT nan values.

nan ÷ 3000 = nan
nan × 255  = nan
nan cast to uint8 = random garbage value

Fix For Future Reference

Add one line after loading the image:
img = np.load(f"{src_dir}/{filename}")

Add this line:  

img = np.nan_to_num(img, nan=0.0)  # replace nan with 0

rgb = img[[2, 1, 0], :, :]

This replaces all nan values with 0 before processing. Clean fix. Remember it for your Dataset class code later.

Task C Dataset Complete

Positive patches: 1,356 images + 1,356 txt files

Negative patches: 2,000 images + 2,000 txt files

─────────────────────────────────────────────────
Total:            3,356 files ready for YOLOv8

# YOLOv8


YOLO Folder Structure

YOLOv8 expects a very specific folder structure. Not optional — if folders are named wrong YOLOv8 cannot find your data.

Required structure:

graveyards_yolo/

├── images/
│   ├── train/    ← 70% of all patches
│   ├── val/      ← 10% of all patches
│   └── test/     ← 20% of all patches
├── labels/
│   ├── train/    ← matching .txt files
│   ├── val/      ← matching .txt files
│   └── test/     ← matching .txt files
└── dataset.yaml  ← tells YOLOv8 where everything is


Train = 2,349 patches-70%

Val   =   336 patches-10%

Test  =   671 patches-20%

Total = 3,356 patches

In [14]:
import os, shutil, random, time

# Source folders
pos_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/positive"
neg_dir = "/content/drive/MyDrive/GeoVision exports/graveyards/patches/negative"

# Destination — YOLO structure
yolo_dir = "/content/drive/MyDrive/GeoVision exports/graveyards_yolo"

# Create all folders
for split in ["train", "val", "test"]:
    os.makedirs(f"{yolo_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{yolo_dir}/labels/{split}", exist_ok=True)

print("Folders created")

# Collect all image files with their matching labels
all_pairs = []

# Add positives
for f in os.listdir(pos_dir):
    if f.endswith('.jpg'):
        img_path = f"{pos_dir}/{f}"
        lbl_path = f"{pos_dir}/{f.replace('.jpg', '.txt')}"
        if os.path.exists(lbl_path):
            all_pairs.append((img_path, lbl_path))

# Add negatives
for f in os.listdir(neg_dir):
    if f.endswith('.jpg'):
        img_path = f"{neg_dir}/{f}"
        lbl_path = f"{neg_dir}/{f.replace('.jpg', '.txt')}"
        if os.path.exists(lbl_path):
            all_pairs.append((img_path, lbl_path))

print(f"Total pairs found: {len(all_pairs)}")

# Shuffle with fixed seed
random.seed(42)
random.shuffle(all_pairs)

# Split
n         = len(all_pairs)
train_end = int(0.70 * n)
val_end   = int(0.80 * n)

splits = {
    "train": all_pairs[:train_end],
    "val"  : all_pairs[train_end:val_end],
    "test" : all_pairs[val_end:]
}

# Copy files into YOLO structure
start = time.time()

for split_name, pairs in splits.items():
    for idx, (img_path, lbl_path) in enumerate(pairs):

        # New filename — sequential numbering
        new_name = f"{split_name}_{idx:05d}"

        # Copy image
        shutil.copy(
            img_path,
            f"{yolo_dir}/images/{split_name}/{new_name}.jpg"
        )

        # Copy label
        shutil.copy(
            lbl_path,
            f"{yolo_dir}/labels/{split_name}/{new_name}.txt"
        )

    elapsed    = time.time() - start
    mins, secs = divmod(int(elapsed), 60)
    print(f"[{mins:02d}:{secs:02d}] {split_name}: {len(pairs)} pairs copied")

print(f"\nDone")

Folders created
Total pairs found: 3356
[09:43] train: 2349 pairs copied
[11:16] val: 335 pairs copied
[14:11] test: 672 pairs copied

Done


Task A split used seed(42):
→ Shuffled Task A tiles
→ Selected train/val/test from Task A

Task C split uses seed(42):
→ Shuffles Task C patches
→ Completely different files
→ No overlap possible

Data leakage only happens when:
SAME files appear in both
training AND test sets

Task A files → .npy satellite tiles
Task C files → .jpg graveyard patches

Completely different files
Different datasets
seed(42) is fine here

The Rule To Remember

Use different seed when:
→ Sampling from SAME pool of files
→ Risk of same file appearing twice
→ Like negatives from Task A tiles

Same seed is fine when:
→ Completely different files
→ No overlap possible
→ Like Task A tiles vs Task C patches

# Create dataset.yaml

dataset.yaml tells YOLOv8:

1. Where your images are
2. Where your labels are
3. How many classes you have
4. What each class is called

Without it YOLOv8 asks:

"Where is my data?"
And crashes immediately

In [15]:
yaml_content = f"""
path: /content/drive/MyDrive/GeoVision exports/graveyards_yolo
train: images/train
val:   images/val
test:  images/test

nc: 1
names: ['graveyard']
"""

yaml_path = f"{yolo_dir}/dataset.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("dataset.yaml created")
print(yaml_content)

dataset.yaml created

path: /content/drive/MyDrive/GeoVision exports/graveyards_yolo
train: images/train
val:   images/val
test:  images/test

nc: 1
names: ['graveyard']



Final Status

339 graveyard locations from OSM

1,356 positive patches + annotations

2,000 negative patches + annotations

YOLO folder structure created

dataset.yaml created

Train/val/test split done

graveyards_yolo/

├── images/
│   ├── train/  → 2,349 images
│   ├── val/    →   335 images
│   └── test/   →   672 images
├── labels/
│   ├── train/  → 2,349 txt files
│   ├── val/    →   335 txt files
│   └── test/   →   672 txt files
└── dataset.yaml